# Faza 3b — Enkoderski LLM (BERTić + mBERT) na Google Colab (BESPLATNO)

Fine-tuning **BERTić** (monolingvalni) i **mBERT** (multilingvalni) za detekciju klikbejta.
Protokol iz postavke: **10-slojna stratifikovana CV** + varijante po **broju epoha** (2/3/4).
Metrike su identične baseline-u radi uporedne tabele.

### ⚠️ Pre pokretanja
1. `Runtime → Change runtime type → Hardware accelerator: **T4 GPU**` (besplatno).
2. Pokreći ćelije **redom**, od vrha.

### 🔒 Bez pristupa nalozima
Notebook **ne traži** pristup Google Drive-u, Dropbox-u ni bilo kom nalogu. Samo klonira javni GitHub repo i skida modele sa HuggingFace-a (nužno za fine-tuning). Rezultate na kraju **ispisuje** da ih kopiraš.

### Vreme (besplatan T4)
Po modelu ≈ 45–90 min (10 fold × 3 epohe), oba ≈ 1.5–3 h. Naslovi su kratki (`--max-len 64`) → brzo.

In [ ]:
# 1) Provera GPU-a
!nvidia-smi -L || echo '❌ Nema GPU — Runtime → Change runtime type → T4 GPU'

In [ ]:
# 2) Klon na ČISTO (briše prethodne klonove → nema ugnežđavanja /OPJ/OPJ/OPJ).
#    ROOT se pamti kao globalna promenljiva za naredne ćelije.
import os, glob, subprocess
subprocess.run('rm -rf /content/repo', shell=True)
subprocess.run('git clone -q https://github.com/filip939/OPJ_ClikcBait.git /content/repo', shell=True)
hits = glob.glob('/content/repo/**/src/transformers/finetune.py', recursive=True)
assert hits, '❌ Ne nalazim finetune.py — da li je repo javan i ima li poslednji push?'
ROOT = hits[0].split('/src/transformers/')[0]
os.chdir(ROOT)
print('ROOT =', ROOT)
!head -2 data/annotated/dataset.tsv && wc -l data/annotated/dataset.tsv

In [ ]:
# 3) Instalacija (Colab već ima torch sa CUDA)
!pip -q install -U transformers datasets accelerate scikit-learn

## Smoke-test (2 fold-a, 1 varijanta epoha) — potvrda da sve radi

In [ ]:
import subprocess
subprocess.run(f'cd {ROOT} && python src/transformers/finetune.py --model bertic --epochs 3 --quick', shell=True)

## Pun run — BERTić, pa mBERT (10-fold × epohe 2/3/4)
Rezultati: `results/encoder_bertic_results.csv`, `results/encoder_mbert_results.csv`.

In [ ]:
import subprocess
subprocess.run(f'cd {ROOT} && python src/transformers/finetune.py --model bertic --epochs 2 3 4', shell=True)
subprocess.run(f'cd {ROOT} && python src/transformers/finetune.py --model mbert  --epochs 2 3 4', shell=True)

In [ ]:
# 4) Ispiši rezultate — KOPIRAJ izlaz (bez pristupa nalozima).
import glob
for f in sorted(glob.glob(f'{ROOT}/results/encoder_*_results.csv')):
    print('\n===== ', f, ' ====='); print(open(f).read())